In [ ]:
# Install the original QLoRA stack and the shared RAG evaluation packages.
!pip install -q -U "transformers>=4.48,<5" "datasets>=3.0" "accelerate>=1.0" "peft>=0.14" "trl>=0.24,<1" "bitsandbytes>=0.45" "huggingface_hub>=0.27" "pandas>=2.0" "tqdm>=4.66" rapidfuzz sacrebleu bert-score==0.3.13


In [ ]:
# Import the original fine-tuning stack and set reproducibility.
import json, random, re, unicodedata, numpy as np, pandas as pd, torch
from pathlib import Path
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score
SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this QLoRA notebook.")
print(torch.cuda.get_device_name(0))


In [ ]:
# Log in to Hugging Face before loading the gated Llama model.
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# Keep the original fine-tuning configuration and requested GitHub data paths.
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
TRAIN_URL = "https://raw.githubusercontent.com/RamijWasithRahat/Bangla-Agriculture-Chatbot/main/Data/Bangla_Agriculture_QA_Train_800.json"
TEST_URL = "https://raw.githubusercontent.com/RamijWasithRahat/Bangla-Agriculture-Chatbot/main/Data/Bangla_Agriculture_QA_Test_200.json"
OUTPUT_DIR = "./llama-3.2-1b-bangla-agriculture-qlora"
ADAPTER_DIR = "./llama-3.2-1b-bangla-agriculture-adapter"
RESULT_DIR = Path("./finetuned_results"); RESULT_DIR.mkdir(exist_ok=True)
MAX_LENGTH, NUM_EPOCHS, TRAIN_BATCH_SIZE = 512, 5, 4
GRAD_ACCUM_STEPS, LEARNING_RATE = 4, 2e-4
GENERATION_BATCH_SIZE, MAX_NEW_TOKENS = 8, 160
SYSTEM_PROMPT = "তুমি বাংলাদেশের কৃষি বিষয়ক প্রশ্নের উত্তর দেওয়ার জন্য একটি সহায়ক সহকারী। শুধু প্রশ্নের প্রাসঙ্গিক উত্তর বাংলায় দাও। যেখানে সংখ্যা, সময়, দূরত্ব, পরিমাণ বা নির্দিষ্ট তথ্য আছে সেখানে তা সঠিকভাবে উল্লেখ করো। অপ্রয়োজনীয় তথ্য তৈরি করো না।"


In [ ]:
# Load all 800 training examples and the untouched 200-example test set.
train_raw = load_dataset("json", data_files=TRAIN_URL, field="qa_pairs", split="train")
test_raw = load_dataset("json", data_files=TEST_URL, field="qa_pairs", split="train")
assert len(train_raw) == 800 and len(test_raw) == 200
print(train_raw, test_raw)


In [ ]:
# Convert the original QA records into prompt-completion training format.
def to_sft_format(example):
    return {
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": example["question"].strip()}],
        "completion": [{"role": "assistant", "content": example["reference_answer"].strip()}],
    }
train_sft = train_raw.map(to_sft_format, remove_columns=train_raw.column_names)
print(train_sft[0])


In [ ]:
# Load the tokenizer with the original right-padding training setup.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(tokenizer.pad_token, tokenizer.eos_token)


In [ ]:
# Check sequence lengths without changing the original maximum length.
def formatted_length(example):
    text = tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False, add_generation_prompt=False)
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])
lengths = [formatted_length(x) for x in train_sft]
print("max:", max(lengths), "over limit:", sum(x > MAX_LENGTH for x in lengths))


In [ ]:
# Configure the original 4-bit NF4 QLoRA quantization.
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
print(compute_dtype)


In [ ]:
# Load and prepare the original base model for k-bit training.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto", dtype=compute_dtype)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print("4-bit:", getattr(model, "is_loaded_in_4bit", False))


In [ ]:
# Keep the original LoRA adapter configuration unchanged.
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
print(peft_config)


In [ ]:
# Keep the original five-epoch SFT training arguments unchanged.
training_args = SFTConfig(
    output_dir=OUTPUT_DIR, num_train_epochs=NUM_EPOCHS, per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS, learning_rate=LEARNING_RATE, weight_decay=0.01,
    warmup_ratio=0.05, lr_scheduler_type="cosine", max_grad_norm=0.3, max_length=MAX_LENGTH,
    completion_only_loss=True, packing=False, gradient_checkpointing=True, bf16=use_bf16,
    fp16=not use_bf16, optim="paged_adamw_8bit", save_strategy="epoch", save_total_limit=2,
    logging_steps=10, seed=SEED, data_seed=SEED, report_to="none",
)
print(training_args)


In [ ]:
# Create the original SFTTrainer with the LoRA adapter.
trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_sft, processing_class=tokenizer, peft_config=peft_config)
trainer.model.print_trainable_parameters()


In [ ]:
# Fine-tune Llama 3.2 1B on the 800 agriculture examples.
train_result = trainer.train()
print(train_result.metrics)


In [ ]:
# Save the trained LoRA adapter and tokenizer.
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved:", ADAPTER_DIR)


In [ ]:
# Restore the original deterministic generation setup.
trainer.model.config.use_cache = True
trainer.model.eval()
terminators = [tokenizer.eos_token_id]
eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if isinstance(eot_id, int) and eot_id >= 0 and eot_id not in terminators:
    terminators.append(eot_id)
def make_generation_prompt(question):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question.strip()}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:
# Generate one test answer with the original decoding settings.
def generate_one(question, max_new_tokens=MAX_NEW_TOKENS):
    old_side = tokenizer.padding_side; tokenizer.padding_side = "left"
    inputs = tokenizer(make_generation_prompt(question), return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.inference_mode():
        output = trainer.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=terminators, pad_token_id=tokenizer.pad_token_id)
    generated = output[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    tokenizer.padding_side = old_side
    return answer
print(generate_one(test_raw[0]["question"]))


In [ ]:
# Generate all 200 fine-tuned predictions and track max-length truncation.
from tqdm.auto import tqdm
def generate_batch(questions, max_new_tokens=MAX_NEW_TOKENS):
    old_side = tokenizer.padding_side; tokenizer.padding_side = "left"
    prompts = [make_generation_prompt(q) for q in questions]
    batch = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    width = batch["input_ids"].shape[1]
    with torch.inference_mode():
        outputs = trainer.model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=terminators, pad_token_id=tokenizer.pad_token_id)
    answers, flags = [], []
    for output in outputs:
        generated = output[width:]
        ids = generated.tolist()
        answers.append(tokenizer.decode(generated, skip_special_tokens=True).strip())
        flags.append(len(ids) >= max_new_tokens and not any(t in terminators for t in ids))
    tokenizer.padding_side = old_side
    return answers, flags
all_predictions, all_truncated = [], []
for start in tqdm(range(0, len(test_raw), GENERATION_BATCH_SIZE)):
    preds, flags = generate_batch(test_raw[start:start+GENERATION_BATCH_SIZE]["question"])
    all_predictions.extend(preds); all_truncated.extend(flags)
assert len(all_predictions) == len(test_raw) == 200


In [ ]:
# Build the fine-tuned prediction table.
predictions_df = pd.DataFrame({
    "id": test_raw["id"],
    "question": test_raw["question"],
    "gold": test_raw["reference_answer"],
    "prediction": all_predictions,
    "truncated": all_truncated,
})
display(predictions_df.head())


In [ ]:
BN_TO_EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
def normalize(text):
    text = unicodedata.normalize("NFKC", str(text)).translate(BN_TO_EN).lower()
    text = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()
def tokens(text):
    return normalize(text).split()


In [ ]:
# Use the same Exact Match and Token F1 definitions.
def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))
def token_f1(pred, gold):
    p, g = tokens(pred), tokens(gold)
    if not p or not g:
        return 0.0
    overlap = sum((Counter(p) & Counter(g)).values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(g)
    return 2 * precision * recall / (precision + recall)


In [ ]:
# Use the same ROUGE-1 and ROUGE-2 definitions as RAG.
def rouge_n(pred, gold, n):
    p, g = tokens(pred), tokens(gold)
    if len(p) < n or len(g) < n:
        return 0.0
    pg = Counter(tuple(p[i:i+n]) for i in range(len(p)-n+1))
    gg = Counter(tuple(g[i:i+n]) for i in range(len(g)-n+1))
    overlap = sum((pg & gg).values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / sum(pg.values()), overlap / sum(gg.values())
    return 2 * precision * recall / (precision + recall)


In [ ]:
# Use the same ROUGE-L definition as RAG.
def rouge_l(pred, gold):
    p, g = tokens(pred), tokens(gold)
    if not p or not g:
        return 0.0
    dp = [0] * (len(g) + 1)
    for x in p:
        new = [0]
        for j, y in enumerate(g, 1):
            new.append(dp[j-1] + 1 if x == y else max(dp[j], new[-1]))
        dp = new
    lcs = dp[-1]
    precision, recall = lcs / len(p), lcs / len(g)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


In [ ]:
# Calculate all row-level metrics with the exact RAG definitions.
df = predictions_df.copy().fillna("")
df["Exact Match"] = [exact_match(p, g) for p, g in zip(df["prediction"], df["gold"])]
df["Fuzzy Match"] = [fuzz.token_set_ratio(normalize(p), normalize(g)) / 100 for p, g in zip(df["prediction"], df["gold"])]
df["Token F1"] = [token_f1(p, g) for p, g in zip(df["prediction"], df["gold"])]
df["ROUGE-1"] = [rouge_n(p, g, 1) for p, g in zip(df["prediction"], df["gold"])]
df["ROUGE-2"] = [rouge_n(p, g, 2) for p, g in zip(df["prediction"], df["gold"])]
df["ROUGE-L"] = [rouge_l(p, g) for p, g in zip(df["prediction"], df["gold"])]


In [ ]:
# Calculate Corpus BLEU with the exact RAG SacreBLEU configuration.
bleu = BLEU(tokenize="none", smooth_method="exp", effective_order=True)
pred_texts = [" ".join(tokens(x)) for x in df["prediction"]]
gold_texts = [" ".join(tokens(x)) for x in df["gold"]]
corpus_bleu = bleu.corpus_score(pred_texts, [gold_texts]).score / 100
print("Corpus BLEU:", corpus_bleu)


In [ ]:
# Calculate multilingual BERTScore with the exact RAG model and options.
P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(), df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased", batch_size=4, device="cpu",
    idf=False, rescale_with_baseline=False, verbose=True,
)
df["BERT Precision"], df["BERT Recall"], df["BERT F1"] = P.cpu().numpy(), R.cpu().numpy(), F1.cpu().numpy()


In [ ]:
# Build the final fine-tuned result table in the same order as RAG.
result = pd.DataFrame({
    "metric": ["Exact Match", "Fuzzy Match", "Corpus BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "Token F1", "BERT Precision", "BERT Recall", "BERT F1", "Truncated Outputs"],
    "score": [df["Exact Match"].mean(), df["Fuzzy Match"].mean(), corpus_bleu, df["ROUGE-1"].mean(), df["ROUGE-2"].mean(), df["ROUGE-L"].mean(), df["Token F1"].mean(), df["BERT Precision"].mean(), df["BERT Recall"].mean(), df["BERT F1"].mean(), df["truncated"].astype(str).str.lower().eq("true").sum()],
})
display(result)


In [ ]:
# Save the fine-tuned row-level predictions and final comparable result CSV.
df.to_csv(RESULT_DIR / "finetuned_predictions.csv", index=False, encoding="utf-8-sig")
result.to_csv(RESULT_DIR / "finetuned_result.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(trainer.state.log_history).to_csv(RESULT_DIR / "training_log.csv", index=False, encoding="utf-8-sig")
print("Saved to:", RESULT_DIR.resolve())
